# Module 03 — Unsupervised & Statistical Learning
## 03-06: Gaussian Mixture Models & EM Algorithm

**Objective:** Derive and implement the Expectation-Maximization (EM) algorithm from scratch for fitting Gaussian Mixture Models, and connect the algorithm to K-Means as a special case.

**Prerequisites:** 1-07 (Probability & Statistics), 3-01 (K-Means Clustering)

## Part 0 — Setup & Prerequisites

This notebook covers Gaussian Mixture Models (GMMs) and the Expectation-Maximization (EM) algorithm. We will implement the full EM loop from scratch using NumPy, build a reusable `GaussianMixtureModel` class, apply it to synthetic clustered data, and compare it against sklearn's implementation.

**Prerequisites:** 1-07 (Probability & Statistics), 3-01 (K-Means Clustering)

In [ ]:
import sys
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture as SklearnGMM
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler
import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Experiment constants ──────────────────────────────────────────────────────
N_SAMPLES   = 600
K_TRUE      = 4
N_DIMS      = 2
MAX_ITER_EM = 300
TOL_EM      = 1e-4
N_INIT      = 5
K_MAX       = 8
REG_COV     = 1e-6
CLUSTER_STD = [0.8, 1.2, 0.6, 1.0]

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = list(plt.cm.tab10.colors)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('Configuration loaded.')

### Data Loading & EDA

We generate a 4-cluster dataset using `make_blobs` with deliberately varied cluster standard deviations. Clusters of different sizes and spreads test whether EM can discover heterogeneous Gaussian components — something K-Means struggles with when clusters have different covariance structures.

In [ ]:
# ── Generate dataset ──────────────────────────────────────────────────────────
centers = np.array([[-4.0, -4.0], [-4.0, 4.0], [4.0, -4.0], [4.0, 4.0]])
X_raw, y_true = make_blobs(
    n_samples=N_SAMPLES,
    centers=centers,
    cluster_std=CLUSTER_STD,
    random_state=SEED,
)

# Standardize for numerical stability
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Dataset shape  : {X.shape}')
print(f'Class counts   : {dict(zip(*np.unique(y_true, return_counts=True)))}')
print(f'Feature range  : [{X.min():.3f}, {X.max():.3f}]')
print(f'Cluster stds   : {CLUSTER_STD}')

# ── Cluster statistics ────────────────────────────────────────────────────────
rows = []
for k in range(K_TRUE):
    mask = y_true == k
    Xk = X[mask]
    rows.append({
        'Cluster': k,
        'n': int(mask.sum()),
        'mean_x': round(float(Xk[:, 0].mean()), 3),
        'mean_y': round(float(Xk[:, 1].mean()), 3),
        'std_x':  round(float(Xk[:, 0].std()),  3),
        'std_y':  round(float(Xk[:, 1].std()),  3),
    })
df_stats = pd.DataFrame(rows).set_index('Cluster')
print('\nPer-cluster statistics (after standardisation):')
print(df_stats.to_string())

# ── Scatter EDA ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for k in range(K_TRUE):
    mask = y_true == k
    ax.scatter(X[mask, 0], X[mask, 1], c=[COLORS[k]], alpha=0.6,
               s=25, label=f'Cluster {k} (std={CLUSTER_STD[k]})')
ax.set_title('Ground Truth Labels', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(X[:, 0], X[:, 1], c='steelblue', alpha=0.4, s=20)
ax.set_title('Unlabelled Data (Unsupervised Setting)', fontsize=13)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

fig.suptitle('Synthetic 4-Cluster Dataset — make_blobs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 1 — Gaussian Mixture Models & EM from Scratch

### 1.1 The GMM Generative Model

A Gaussian Mixture Model with $K$ components defines a probability density:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

where $\pi_k \ge 0$, $\sum_k \pi_k = 1$ are the **mixing weights**, $\boldsymbol{\mu}_k$ is the mean, and $\boldsymbol{\Sigma}_k$ is the covariance of component $k$.

The **log-likelihood** of the dataset $\{\mathbf{x}_n\}_{n=1}^N$ is:

$$\log p(X) = \sum_{n=1}^{N} \log \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

The sum inside the log makes direct maximization intractable — this is exactly the problem EM solves.

### 1.2 The EM Algorithm

EM introduces a **latent variable** $z_n \in \{1,\ldots,K\}$ indicating component membership. By optimising a lower bound (ELBO) on the log-likelihood:

**E-step** — compute soft assignments (responsibilities):
$$r_{nk} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_j \pi_j \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**M-step** — re-estimate parameters using weighted statistics:
$$N_k = \sum_n r_{nk}, \quad \pi_k = \frac{N_k}{N}, \quad \boldsymbol{\mu}_k = \frac{1}{N_k}\sum_n r_{nk}\mathbf{x}_n$$
$$\boldsymbol{\Sigma}_k = \frac{1}{N_k}\sum_n r_{nk}(\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^\top$$

EM guarantees the log-likelihood is non-decreasing at every step.

### 1.3 Numerical Stability

We work in log-space throughout. The log-sum-exp trick avoids underflow:
$$\log\sum_k \exp(a_k) = a^* + \log\sum_k \exp(a_k - a^*), \quad a^* = \max_k a_k$$

For the multivariate Gaussian, we use **Cholesky decomposition** to compute $\log|\boldsymbol{\Sigma}|$ and the Mahalanobis distance without direct matrix inversion.

In [ ]:
def log_multivariate_gaussian_pdf(
    X: np.ndarray,
    mu: np.ndarray,
    Sigma: np.ndarray,
    reg: float = REG_COV,
) -> np.ndarray:
    '''Compute log-density of a multivariate Gaussian via Cholesky decomposition.

    Uses the Cholesky factor L of (Sigma + reg*I) to compute log|Sigma| and
    squared Mahalanobis distances without direct inversion.

    Args:
        X:    Data matrix of shape (n, d).
        mu:   Mean vector of shape (d,).
        Sigma: Covariance matrix of shape (d, d).
        reg:  Regularisation added to the diagonal.

    Returns:
        Log-densities of shape (n,).
    '''
    d = X.shape[1]
    Sigma_reg = Sigma + reg * np.eye(d)
    try:
        L = np.linalg.cholesky(Sigma_reg)
    except np.linalg.LinAlgError:
        # Increase regularisation if Cholesky fails
        L = np.linalg.cholesky(Sigma + 1e-3 * np.eye(d))
    log_det  = 2.0 * np.sum(np.log(np.diag(L)))
    diff     = X - mu                        # (n, d)
    z        = np.linalg.solve(L, diff.T)   # (d, n)  solves L z = diff^T
    maha_sq  = np.sum(z ** 2, axis=0)       # (n,)  ||L^{-1}(x-mu)||^2
    log_pdf  = -0.5 * (d * np.log(2.0 * np.pi) + log_det + maha_sq)
    return log_pdf

# ── Quick sanity check against scipy ─────────────────────────────────────────
rng_test = np.random.RandomState(0)
mu_test    = np.array([1.0, 2.0])
Sigma_test = np.array([[2.0, 0.5], [0.5, 1.0]])
X_test     = rng_test.randn(5, 2)
ours   = log_multivariate_gaussian_pdf(X_test, mu_test, Sigma_test)
ref    = multivariate_normal(mean=mu_test, cov=Sigma_test).logpdf(X_test)
max_err = np.max(np.abs(ours - ref))
print(f'log_mvn max abs error vs scipy: {max_err:.2e}')
assert max_err < 1e-8, 'Cholesky log-pdf deviates from scipy reference!'
print('Cholesky log-PDF matches scipy reference.')

### 1.4 E-Step: Computing Responsibilities

The E-step computes $r_{nk}$ for every data point and component. All computations are in log-space to avoid numerical underflow; we use `scipy.special.logsumexp` for the normalisation denominator.

In [ ]:
def e_step(
    X: np.ndarray,
    log_pi: np.ndarray,
    mus: list,
    Sigmas: list,
    reg: float = REG_COV,
) -> tuple:
    '''Compute log-responsibilities for all data points (E-step).

    log r_{nk} = log pi_k + log N(x_n | mu_k, Sigma_k) - log sum_j(...)

    Args:
        X:       Data matrix of shape (n, d).
        log_pi:  Log mixing weights of shape (K,).
        mus:     List of K mean vectors, each of shape (d,).
        Sigmas:  List of K covariance matrices, each of shape (d, d).
        reg:     Covariance regularisation.

    Returns:
        log_R:   Log-responsibility matrix of shape (n, K).
        log_lik: Total log-likelihood (scalar float).
    '''
    n = X.shape[0]
    K = len(mus)
    log_weighted = np.zeros((n, K))
    for k in range(K):
        log_weighted[:, k] = (
            log_pi[k] + log_multivariate_gaussian_pdf(X, mus[k], Sigmas[k], reg=reg)
        )
    # log-sum-exp across K components
    log_sum = logsumexp(log_weighted, axis=1)     # (n,)
    log_R   = log_weighted - log_sum[:, np.newaxis]  # (n, K) normalised
    log_lik = float(np.sum(log_sum))
    return log_R, log_lik

### 1.5 M-Step: Updating Parameters

The M-step re-estimates $\pi_k$, $\boldsymbol{\mu}_k$, $\boldsymbol{\Sigma}_k$ by maximising the expected complete-data log-likelihood. The update equations are closed-form weighted statistics using the responsibilities computed in the E-step.

In [ ]:
def m_step(
    X: np.ndarray,
    log_R: np.ndarray,
    reg: float = REG_COV,
) -> tuple:
    '''Re-estimate GMM parameters from soft assignments (M-step).

    Updates are weighted statistics:
        N_k    = sum_n r_{nk}
        pi_k   = N_k / N
        mu_k   = (1/N_k) sum_n r_{nk} x_n
        Sig_k  = (1/N_k) sum_n r_{nk} (x_n - mu_k)(x_n - mu_k)^T + reg*I

    Args:
        X:     Data matrix of shape (n, d).
        log_R: Log-responsibility matrix of shape (n, K).
        reg:   Regularisation added to covariance diagonal.

    Returns:
        log_pi: Updated log mixing weights of shape (K,).
        mus:    Updated list of K means, each of shape (d,).
        Sigmas: Updated list of K covariances, each of shape (d, d).
    '''
    n, d = X.shape
    K    = log_R.shape[1]
    R    = np.exp(log_R)                        # (n, K) responsibilities
    N_k  = np.maximum(R.sum(axis=0), 1e-10)    # (K,)  effective counts
    log_pi = np.log(N_k) - np.log(n)           # (K,)  updated log weights
    mus = [(R[:, k] @ X) / N_k[k] for k in range(K)]
    Sigmas = []
    for k in range(K):
        diff    = X - mus[k]               # (n, d)
        w       = R[:, k] / N_k[k]        # (n,)  normalised weights
        Sigma_k = (diff.T * w) @ diff     # (d, d) weighted outer product
        Sigma_k += reg * np.eye(d)        # diagonal regularisation
        Sigmas.append(Sigma_k)
    return log_pi, mus, Sigmas

### 1.6 Log-Likelihood, Initialisation & Full EM Loop

We need three more building blocks:
- `compute_log_likelihood` — evaluates $\log p(X)$ given current parameters
- `initialize_gmm_params` — sets up starting values via random selection or K-Means
- `run_em_with_history` — runs the full E→M→E→M... loop and records snapshots

In [ ]:
def compute_log_likelihood(
    X: np.ndarray,
    log_pi: np.ndarray,
    mus: list,
    Sigmas: list,
    reg: float = REG_COV,
) -> float:
    '''Evaluate the GMM log-likelihood on dataset X.

    log p(X) = sum_n log sum_k pi_k N(x_n | mu_k, Sigma_k)

    Args:
        X:       Data matrix of shape (n, d).
        log_pi:  Log mixing weights of shape (K,).
        mus:     List of K means, each of shape (d,).
        Sigmas:  List of K covariance matrices, each of shape (d, d).
        reg:     Covariance regularisation.

    Returns:
        Total log-likelihood as a Python float.
    '''
    n, K  = X.shape[0], len(mus)
    log_w = np.zeros((n, K))
    for k in range(K):
        log_w[:, k] = (
            log_pi[k] + log_multivariate_gaussian_pdf(X, mus[k], Sigmas[k], reg=reg)
        )
    return float(np.sum(logsumexp(log_w, axis=1)))


def initialize_gmm_params(
    X: np.ndarray,
    K: int,
    method: str = 'kmeans',
    random_state: int = SEED,
    reg: float = REG_COV,
) -> tuple:
    '''Initialise GMM parameters using random or K-Means strategy.

    'random': picks K random data points as means; sets isotropic covariances
    scaled by the global variance; uses uniform mixing weights.

    'kmeans': runs K-Means to get centroid starting points; estimates per-
    cluster sample covariances from hard assignments; uses empirical weights.

    Args:
        X:            Data matrix of shape (n, d).
        K:            Number of mixture components.
        method:       'random' or 'kmeans'.
        random_state: Random seed.
        reg:          Covariance regularisation.

    Returns:
        log_pi: Initial log mixing weights of shape (K,).
        mus:    Initial list of K means, each of shape (d,).
        Sigmas: Initial list of K covariances, each of shape (d, d).
    '''
    rng   = np.random.RandomState(random_state)
    n, d  = X.shape
    if method == 'random':
        idx    = rng.choice(n, size=K, replace=False)
        mus    = [X[idx[k]].copy() for k in range(K)]
        scale  = float(np.var(X))
        Sigmas = [scale * np.eye(d) + reg * np.eye(d) for _ in range(K)]
        log_pi = np.full(K, -np.log(K))
    elif method == 'kmeans':
        km     = KMeans(n_clusters=K, random_state=random_state, n_init=10)
        labels = km.fit_predict(X)
        mus    = [km.cluster_centers_[k].copy() for k in range(K)]
        Sigmas = []
        N_k    = np.zeros(K)
        for k in range(K):
            mask    = labels == k
            N_k[k]  = mask.sum()
            if mask.sum() > d + 1:
                diff    = X[mask] - mus[k]
                Sigma_k = (diff.T @ diff) / mask.sum() + reg * np.eye(d)
            else:
                Sigma_k = float(np.var(X)) * np.eye(d) + reg * np.eye(d)
            Sigmas.append(Sigma_k)
        log_pi = np.log(np.maximum(N_k, 1e-10)) - np.log(n)
    else:
        raise ValueError(f'Unknown method: {method!r}. Use "random" or "kmeans".')
    return log_pi, mus, Sigmas

In [ ]:
def run_em_with_history(
    X: np.ndarray,
    K: int,
    init_method: str = 'kmeans',
    max_iter: int = MAX_ITER_EM,
    tol: float = TOL_EM,
    random_state: int = SEED,
    reg: float = REG_COV,
    snapshot_iters: tuple = (0, 1, 2, 5, 10, 20, 50),
) -> dict:
    '''Run EM for a K-component GMM and record the full optimisation history.

    Records parameter snapshots at selected iterations for visualising
    how the model evolves during training.

    Args:
        X:              Data matrix of shape (n, d).
        K:              Number of mixture components.
        init_method:    Initialisation strategy ('random' or 'kmeans').
        max_iter:       Maximum EM iterations.
        tol:            Convergence tolerance on log-likelihood change.
        random_state:   Random seed.
        reg:            Covariance regularisation.
        snapshot_iters: Iterations at which to save parameter snapshots.

    Returns:
        Dict with keys: log_pi, mus, Sigmas, log_likelihoods,
        snapshots, n_iter, final_ll.
    '''
    log_pi, mus, Sigmas = initialize_gmm_params(
        X, K, method=init_method, random_state=random_state, reg=reg
    )
    log_likelihoods = []
    snapshots       = {}
    prev_ll         = -np.inf

    for it in range(max_iter):
        log_R, ll = e_step(X, log_pi, mus, Sigmas, reg=reg)
        log_likelihoods.append(ll)

        if it in snapshot_iters:
            snapshots[it] = {
                'log_pi': log_pi.copy(),
                'mus':    [m.copy() for m in mus],
                'Sigmas': [S.copy() for S in Sigmas],
                'log_R':  log_R.copy(),
                'll':     ll,
            }

        delta = abs(ll - prev_ll)
        if it > 0 and delta < tol:
            print(f'  Converged at iteration {it + 1}  (delta={delta:.2e})')
            break
        prev_ll    = ll
        log_pi, mus, Sigmas = m_step(X, log_R, reg=reg)

    # Final E-step to get the terminal log-likelihood
    _, final_ll = e_step(X, log_pi, mus, Sigmas, reg=reg)
    return {
        'log_pi':          log_pi,
        'mus':             mus,
        'Sigmas':          Sigmas,
        'log_likelihoods': log_likelihoods,
        'snapshots':       snapshots,
        'n_iter':          len(log_likelihoods),
        'final_ll':        final_ll,
    }

---
## Part 2 — Putting It All Together

We now assemble the EM components into a reusable `GaussianMixtureModel` class that mirrors the sklearn API. Key features:
- Multiple restarts (`n_init`) to escape local optima — keeps the best result
- BIC and AIC model selection criteria
- `sample()` for generative use

In [ ]:
def count_free_params(K: int, d: int, cov_type: str = 'full') -> int:
    '''Count free parameters in a GMM.

    Args:
        K:        Number of components.
        d:        Data dimensionality.
        cov_type: Covariance type ('full', 'diagonal', 'spherical').

    Returns:
        Total number of free parameters.
    '''
    mean_params = K * d
    if cov_type == 'full':
        cov_params = K * d * (d + 1) // 2
    elif cov_type == 'diagonal':
        cov_params = K * d
    elif cov_type == 'spherical':
        cov_params = K
    else:
        raise ValueError(f'Unknown cov_type: {cov_type!r}')
    mix_params = K - 1
    return mean_params + cov_params + mix_params


class GaussianMixtureModel:
    '''From-scratch GMM fitted via EM with multiple random restarts.

    Supports K-Means or random initialisation. BIC and AIC are computed
    from the best restart for model-order selection.

    Attributes:
        log_pi_:         Log mixing weights after fitting, shape (K,).
        mus_:            List of K fitted mean vectors, each of shape (d,).
        Sigmas_:         List of K fitted covariance matrices, each (d, d).
        log_likelihood_: Best log-likelihood over all restarts.
        n_iter_:         EM iterations for the best restart.
        converged_:      Whether the best restart converged.
    '''

    def __init__(
        self,
        n_components: int = 1,
        max_iter: int = MAX_ITER_EM,
        tol: float = TOL_EM,
        n_init: int = N_INIT,
        init_method: str = 'kmeans',
        reg_covar: float = REG_COV,
        random_state: int = SEED,
    ) -> None:
        '''Store hyperparameters and initialise fitted-attribute placeholders.

        Args:
            n_components:  Number of mixture components K.
            max_iter:      Maximum EM iterations per restart.
            tol:           Log-likelihood convergence tolerance.
            n_init:        Number of independent random restarts.
            init_method:   Initialisation strategy ('kmeans' or 'random').
            reg_covar:     Regularisation added to covariance diagonal.
            random_state:  Master random seed.
        '''
        self.n_components = n_components
        self.max_iter     = max_iter
        self.tol          = tol
        self.n_init       = n_init
        self.init_method  = init_method
        self.reg_covar    = reg_covar
        self.random_state = random_state
        self.log_pi_          = None
        self.mus_             = None
        self.Sigmas_          = None
        self.log_likelihood_  = -np.inf
        self.n_iter_          = 0
        self.converged_       = False

    def fit(self, X: np.ndarray) -> 'GaussianMixtureModel':
        '''Fit GMM parameters via EM with multiple restarts.

        Runs n_init independent EM runs; keeps the parameters achieving
        the highest final log-likelihood.

        Args:
            X: Training data of shape (n, d).

        Returns:
            self — fitted estimator (supports method chaining).
        '''
        rng     = np.random.RandomState(self.random_state)
        best_ll = -np.inf
        for init_idx in range(self.n_init):
            rs = int(rng.randint(0, 10_000))
            log_pi, mus, Sigmas = initialize_gmm_params(
                X, self.n_components,
                method=self.init_method,
                random_state=rs,
                reg=self.reg_covar,
            )
            prev_ll   = -np.inf
            converged = False
            n_it      = 0
            for it in range(self.max_iter):
                log_R, ll = e_step(X, log_pi, mus, Sigmas, reg=self.reg_covar)
                if it > 0 and abs(ll - prev_ll) < self.tol:
                    converged = True
                    n_it      = it + 1
                    break
                prev_ll = ll
                n_it    = it + 1
                log_pi, mus, Sigmas = m_step(X, log_R, reg=self.reg_covar)
            _, final_ll = e_step(X, log_pi, mus, Sigmas, reg=self.reg_covar)
            if final_ll > best_ll:
                best_ll              = final_ll
                self.log_pi_         = log_pi.copy()
                self.mus_            = [m.copy() for m in mus]
                self.Sigmas_         = [S.copy() for S in Sigmas]
                self.log_likelihood_ = final_ll
                self.n_iter_         = n_it
                self.converged_      = converged
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        '''Assign each data point to the most probable component.

        Args:
            X: Data of shape (n, d).

        Returns:
            Integer component labels of shape (n,).
        '''
        log_R, _ = e_step(
            X, self.log_pi_, self.mus_, self.Sigmas_, reg=self.reg_covar
        )
        return np.argmax(log_R, axis=1)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        '''Return soft component membership probabilities.

        Args:
            X: Data of shape (n, d).

        Returns:
            Responsibility matrix of shape (n, K).
        '''
        log_R, _ = e_step(
            X, self.log_pi_, self.mus_, self.Sigmas_, reg=self.reg_covar
        )
        return np.exp(log_R)

    def score(self, X: np.ndarray) -> float:
        '''Compute mean per-sample log-likelihood.

        Args:
            X: Data of shape (n, d).

        Returns:
            Average log-likelihood (higher is better).
        '''
        ll = compute_log_likelihood(
            X, self.log_pi_, self.mus_, self.Sigmas_, reg=self.reg_covar
        )
        return ll / len(X)

    def sample(self, n_samples: int, random_state: int = SEED) -> tuple:
        '''Generate new samples from the fitted distribution.

        Args:
            n_samples:    Number of samples to draw.
            random_state: Random seed.

        Returns:
            X_samp:   Generated samples of shape (n_samples, d).
            comp_ids: Component index per sample, shape (n_samples,).
        '''
        rng      = np.random.RandomState(random_state)
        pi       = np.exp(self.log_pi_)
        pi      /= pi.sum()   # ensure sums to 1
        comp_ids = rng.choice(self.n_components, size=n_samples, p=pi)
        d        = self.mus_[0].shape[0]
        X_samp   = np.zeros((n_samples, d))
        for k in range(self.n_components):
            mask = comp_ids == k
            if mask.sum() > 0:
                X_samp[mask] = rng.multivariate_normal(
                    self.mus_[k], self.Sigmas_[k], size=int(mask.sum())
                )
        return X_samp, comp_ids

    def bic(self, X: np.ndarray) -> float:
        '''Bayesian Information Criterion: BIC = -2 log L + p log n.

        Args:
            X: Data of shape (n, d).

        Returns:
            BIC value (lower is better).
        '''
        n, d = X.shape
        ll   = compute_log_likelihood(
            X, self.log_pi_, self.mus_, self.Sigmas_, reg=self.reg_covar
        )
        p = count_free_params(self.n_components, d, cov_type='full')
        return -2.0 * ll + p * np.log(n)

    def aic(self, X: np.ndarray) -> float:
        '''Akaike Information Criterion: AIC = -2 log L + 2p.

        Args:
            X: Data of shape (n, d).

        Returns:
            AIC value (lower is better).
        '''
        d  = X.shape[1]
        ll = compute_log_likelihood(
            X, self.log_pi_, self.mus_, self.Sigmas_, reg=self.reg_covar
        )
        p = count_free_params(self.n_components, d, cov_type='full')
        return -2.0 * ll + 2.0 * p

### Sanity Check on Toy Data

Before applying to our full dataset, we verify the assembled class on a simple 2-component 1D mixture where the correct answer is known analytically.

In [ ]:
# ── 2-component 1-D mixture sanity check ─────────────────────────────────────
rng_sc   = np.random.RandomState(SEED)
X_toy    = np.vstack([
    rng_sc.randn(100, 1) * 0.5 + 0.0,
    rng_sc.randn(100, 1) * 0.5 + 4.0,
])
true_mus = [0.0, 4.0]

gmm_toy = GaussianMixtureModel(n_components=2, n_init=3, random_state=SEED)
gmm_toy.fit(X_toy)

fitted_mus = sorted([float(m[0]) for m in gmm_toy.mus_])
print(f'True means         : {true_mus}')
print(f'Fitted means       : {[round(m, 3) for m in fitted_mus]}')
print(f'Log-likelihood     : {gmm_toy.log_likelihood_:.2f}')
print(f'Converged          : {gmm_toy.converged_}  ({gmm_toy.n_iter_} iterations)')
assert abs(fitted_mus[0] - 0.0) < 0.3, 'First mean too far from 0'
assert abs(fitted_mus[1] - 4.0) < 0.3, 'Second mean too far from 4'
print('Sanity check passed.')

# ── Visualise 1-D fit ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
x_plot  = np.linspace(-2, 6, 300).reshape(-1, 1)
labels_toy = gmm_toy.predict(X_toy)
for k in range(2):
    mask = labels_toy == k
    ax.hist(X_toy[mask, 0], bins=20, alpha=0.4, color=COLORS[k],
            density=True, label=f'Cluster {k}')
pdf_vals = np.exp(gmm_toy.predict_proba(x_plot)[:, 0]) * 0   # init
for k in range(2):
    pdf_k = np.exp(gmm_toy.log_pi_[k]) * np.exp(
        log_multivariate_gaussian_pdf(x_plot, gmm_toy.mus_[k], gmm_toy.Sigmas_[k])
    )
    ax.plot(x_plot[:, 0], pdf_k, color=COLORS[k], lw=2)
ax.set_title('1-D Sanity Check: 2-Component GMM', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 3 — Model Selection & Comparison

With the GMM class confirmed, we apply it to our main 4-cluster dataset. The central questions are:
1. **How many components?** — BIC and AIC penalise model complexity.
2. **Does initialisation matter?** — Multiple restarts vs single random init.
3. **How does it compare to sklearn?** — Numerical equivalence check.

In [ ]:
def compute_bic_aic_curve(
    X: np.ndarray,
    k_range: range,
    n_init: int = N_INIT,
    random_state: int = SEED,
) -> pd.DataFrame:
    '''Fit a GMM for each K and record BIC, AIC, and log-likelihood.

    Args:
        X:            Data matrix of shape (n, d).
        k_range:      Range of K values to evaluate.
        n_init:       Number of EM restarts per K.
        random_state: Random seed.

    Returns:
        DataFrame with columns K, log_likelihood, BIC, AIC, n_params, n_iter.
    '''
    rows = []
    for K in k_range:
        gmm = GaussianMixtureModel(
            n_components=K, n_init=n_init, random_state=random_state
        )
        gmm.fit(X)
        rows.append({
            'K':              K,
            'log_likelihood': round(gmm.log_likelihood_, 1),
            'BIC':            round(gmm.bic(X), 1),
            'AIC':            round(gmm.aic(X), 1),
            'n_params':       count_free_params(K, X.shape[1]),
            'n_iter':         gmm.n_iter_,
            'converged':      gmm.converged_,
        })
        print(
            f'  K={K}  log-lik={gmm.log_likelihood_:.1f}  '
            f'BIC={gmm.bic(X):.1f}  AIC={gmm.aic(X):.1f}  '
            f'n_iter={gmm.n_iter_}'
        )
    return pd.DataFrame(rows)


print('BIC / AIC sweep over K = 1 .. 8 ...')
df_sel = compute_bic_aic_curve(X, range(1, K_MAX + 1))
best_bic_k = int(df_sel.loc[df_sel['BIC'].idxmin(), 'K'])
best_aic_k = int(df_sel.loc[df_sel['AIC'].idxmin(), 'K'])
print(f'\nBest K by BIC : {best_bic_k}')
print(f'Best K by AIC : {best_aic_k}')
print(f'True K        : {K_TRUE}')
print('\n', df_sel.set_index('K').to_string())

### BIC / AIC Model Selection Curves

BIC imposes a stronger penalty ($p \log n$ vs $2p$) so it typically selects fewer components. Both should identify $K = 4$ as optimal — verifying that EM has correctly recovered the generating distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ks     = df_sel['K'].values
bics   = df_sel['BIC'].values
aics   = df_sel['AIC'].values
lls    = df_sel['log_likelihood'].values

ax = axes[0]
ax.plot(ks, bics, 'o-', color='royalblue', lw=2, label='BIC')
ax.plot(ks, aics, 's--', color='tomato',   lw=2, label='AIC')
ax.axvline(best_bic_k, color='royalblue', linestyle=':', alpha=0.8,
           label=f'Best BIC: K={best_bic_k}')
ax.axvline(best_aic_k, color='tomato',    linestyle=':', alpha=0.8,
           label=f'Best AIC: K={best_aic_k}')
ax.axvline(K_TRUE, color='black', linestyle='--', lw=1.5, alpha=0.6,
           label=f'True K={K_TRUE}')
ax.set_xlabel('Number of Components K', fontsize=11)
ax.set_ylabel('Information Criterion', fontsize=11)
ax.set_title('BIC & AIC vs K', fontsize=13)
ax.set_xticks(ks)
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(ks, lls, 'o-', color='seagreen', lw=2)
ax.axvline(K_TRUE, color='black', linestyle='--', lw=1.5, alpha=0.6,
           label=f'True K={K_TRUE}')
ax.set_xlabel('Number of Components K', fontsize=11)
ax.set_ylabel('Log-Likelihood', fontsize=11)
ax.set_title('Log-Likelihood vs K (always non-decreasing)', fontsize=13)
ax.set_xticks(ks)
ax.legend(fontsize=9)

fig.suptitle('Model Selection: BIC / AIC Favour Simpler Models', fontsize=14,
             fontweight='bold')
plt.tight_layout()
plt.show()

### Local Optima & Multiple Restarts

EM converges to a **local** maximum of the log-likelihood — the solution depends on initialisation. With a single random start, we may land in a poor solution. Running `n_init` restarts and selecting the best result mitigates this.

In [ ]:
# ── Run 15 independent single-restart random inits ────────────────────────────
n_trials   = 15
final_lls  = []
n_iters    = []
converged  = []

for trial in range(n_trials):
    gmm_trial = GaussianMixtureModel(
        n_components=K_TRUE,
        n_init=1,
        init_method='random',
        random_state=SEED + trial * 37,
    )
    gmm_trial.fit(X)
    final_lls.append(gmm_trial.log_likelihood_)
    n_iters.append(gmm_trial.n_iter_)
    converged.append(gmm_trial.converged_)

final_lls = np.array(final_lls)
print(f'Log-likelihood across {n_trials} single-restart runs:')
print(f'  Min  : {final_lls.min():.2f}')
print(f'  Max  : {final_lls.max():.2f}')
print(f'  Mean : {final_lls.mean():.2f}')
print(f'  Std  : {final_lls.std():.2f}')
print(f'  Converged: {sum(converged)}/{n_trials}')

# ── Best result (n_init=5) ────────────────────────────────────────────────────
gmm_best = GaussianMixtureModel(
    n_components=K_TRUE, n_init=5, init_method='random', random_state=SEED
)
gmm_best.fit(X)
print(f'\nBest with n_init=5 (random): {gmm_best.log_likelihood_:.2f}')

gmm_km = GaussianMixtureModel(
    n_components=K_TRUE, n_init=3, init_method='kmeans', random_state=SEED
)
gmm_km.fit(X)
print(f'Best with n_init=3 (kmeans): {gmm_km.log_likelihood_:.2f}')

# ── Plot distribution of log-likelihoods ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(final_lls, bins=8, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(final_lls.max(), color='seagreen', lw=2, linestyle='--',
           label=f'Best single (max={final_lls.max():.1f})')
ax.axvline(gmm_km.log_likelihood_, color='tomato', lw=2, linestyle='-',
           label=f'K-Means init ({gmm_km.log_likelihood_:.1f})')
ax.set_xlabel('Final Log-Likelihood', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'EM Log-Likelihood Distribution over {n_trials} Random Initialisations'
             f' (K={K_TRUE})', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Library Comparison

We verify our implementation against `sklearn.mixture.GaussianMixture`. We expect the log-likelihoods and cluster assignments to match closely.

In [ ]:
# ── Fit sklearn GMM ───────────────────────────────────────────────────────────
skl_gmm = SklearnGMM(
    n_components=K_TRUE,
    covariance_type='full',
    n_init=N_INIT,
    max_iter=MAX_ITER_EM,
    tol=TOL_EM,
    random_state=SEED,
)
skl_gmm.fit(X)
skl_ll     = skl_gmm.score(X) * len(X)
skl_labels = skl_gmm.predict(X)

# ── Our GMM ───────────────────────────────────────────────────────────────────
our_gmm = GaussianMixtureModel(
    n_components=K_TRUE, n_init=N_INIT, random_state=SEED
)
our_gmm.fit(X)
our_ll     = our_gmm.log_likelihood_
our_labels = our_gmm.predict(X)

print('Log-likelihood comparison (K=4):')
print(f'  Our GMM  : {our_ll:.2f}')
print(f'  sklearn  : {skl_ll:.2f}')
print(f'  Relative diff: {abs(our_ll - skl_ll) / abs(skl_ll) * 100:.3f}%')

# ── ARI between the two sets of labels (should be ~1.0) ──────────────────────
ari_vs_sklearn = adjusted_rand_score(skl_labels, our_labels)
print(f'\nARI (our labels vs sklearn labels): {ari_vs_sklearn:.4f}')

# ── Side-by-side scatter ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, labels, title in zip(
    axes,
    [our_labels, skl_labels],
    ['Our GMM', 'sklearn GaussianMixture'],
):
    for k in range(K_TRUE):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=[COLORS[k]], alpha=0.5,
                   s=20, label=f'Cluster {k}')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=8)

fig.suptitle('Cluster Assignments: Our GMM vs sklearn', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── Mean vector comparison ────────────────────────────────────────────────────
print('\nFitted means comparison (sorted by x-coordinate):')
our_mus_sorted = sorted(our_gmm.mus_, key=lambda m: m[0])
skl_mus_sorted = sorted([skl_gmm.means_[k] for k in range(K_TRUE)], key=lambda m: m[0])
for i, (om, sm) in enumerate(zip(our_mus_sorted, skl_mus_sorted)):
    print(f'  Component {i}: ours={np.round(om, 3)}, sklearn={np.round(sm, 3)}, '
          f'diff={np.linalg.norm(om - sm):.4f}')

### K-Means as a Special Case of GMMs

K-Means is the hard-assignment limit of GMMs where:
- Responsibilities are **binary** (0 or 1) — E-step takes the argmax
- Covariances are **fixed isotropic** ($\sigma^2 I$) — M-step only updates means
- This is equivalent to minimising inertia $\sum_n \|\mathbf{x}_n - \boldsymbol{\mu}_{z_n}\|^2$

The soft assignments of GMM reduce to K-Means hard assignments as $\sigma^2 \to 0$.

In [ ]:
def em_hard_assignment(
    X: np.ndarray,
    K: int,
    max_iter: int = 100,
    tol: float = 1e-4,
    random_state: int = SEED,
) -> dict:
    '''EM with hard (binary) assignments — mathematically equivalent to K-Means.

    The E-step assigns each point to its nearest centroid (one-hot
    responsibilities). The M-step updates only the means.

    Args:
        X:            Data matrix of shape (n, d).
        K:            Number of components.
        max_iter:     Maximum iterations.
        tol:          Convergence tolerance on centroid shift.
        random_state: Random seed.

    Returns:
        Dict with keys centers, labels, inertia_history, n_iter.
    '''
    rng      = np.random.RandomState(random_state)
    idx      = rng.choice(len(X), size=K, replace=False)
    centers  = X[idx].copy()
    inertias = []

    for it in range(max_iter):
        # Hard E-step: assign to nearest centroid
        dists   = np.linalg.norm(X[:, np.newaxis] - centers[np.newaxis], axis=2)
        labels  = np.argmin(dists, axis=1)
        inertia = float(np.sum(np.min(dists ** 2, axis=1)))
        inertias.append(inertia)

        # Hard M-step: update means
        new_centers = np.array([
            X[labels == k].mean(axis=0) if (labels == k).sum() > 0 else centers[k]
            for k in range(K)
        ])
        shift   = float(np.max(np.linalg.norm(new_centers - centers, axis=1)))
        centers = new_centers
        if shift < tol:
            break

    return {'centers': centers, 'labels': labels,
            'inertia_history': inertias, 'n_iter': it + 1}


# ── Compare soft EM vs hard EM (K-Means) ─────────────────────────────────────
result_hard = em_hard_assignment(X, K=K_TRUE, random_state=SEED)
result_soft = run_em_with_history(X, K=K_TRUE, init_method='random',
                                  random_state=SEED)

print('Hard EM (K-Means):')
print(f'  Iterations : {result_hard["n_iter"]}')
print(f'  Final inertia: {result_hard["inertia_history"][-1]:.2f}')
print('\nSoft EM (GMM):')
print(f'  Iterations : {result_soft["n_iter"]}')
print(f'  Final log-lik: {result_soft["final_ll"]:.2f}')

ari_hard = adjusted_rand_score(y_true, result_hard['labels'])
ari_soft = adjusted_rand_score(y_true, np.argmax(result_soft['snapshots'].get(
    max(result_soft['snapshots'].keys()), {}).get('log_R',
    np.zeros((len(X), K_TRUE))), axis=1))

gmm_final = GaussianMixtureModel(n_components=K_TRUE, n_init=3, random_state=SEED)
gmm_final.fit(X)
ari_soft2 = adjusted_rand_score(y_true, gmm_final.predict(X))
print(f'\nARI vs ground truth:')
print(f'  Hard EM (K-Means): {ari_hard:.4f}')
print(f'  Soft EM (GMM)    : {ari_soft2:.4f}')

# ── Inertia curve for hard EM ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(result_hard['inertia_history'], 'o-', color='royalblue', ms=4, lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Inertia (within-cluster SS)')
ax.set_title('Hard EM (K-Means) — Inertia Convergence', fontsize=12)

ax = axes[1]
ax.plot(result_soft['log_likelihoods'], 'o-', color='tomato', ms=4, lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Soft EM (GMM) — Log-Likelihood Convergence', fontsize=12)

plt.tight_layout()
plt.show()

---
## Part 4 — Evaluation & Analysis

With the model validated against sklearn and K-Means, we now perform deeper analysis:
1. **Convergence visualisation** — parameter evolution at selected EM snapshots
2. **Covariance type ablation** — diagonal vs spherical vs full covariance
3. **Density estimation** — the fitted GMM as a generative model
4. **Clustering metrics** — ARI, NMI, silhouette

In [ ]:
def plot_gmm_ellipse(
    ax: plt.Axes,
    mu: np.ndarray,
    Sigma: np.ndarray,
    color: str = 'blue',
    n_std: float = 2.0,
    alpha: float = 0.15,
    lw: float = 2.0,
) -> None:
    '''Draw a 2-D Gaussian confidence ellipse on an axes object.

    Computes the ellipse axes and angle from the eigendecomposition of Sigma.
    The ellipse boundary corresponds to n_std standard deviations.

    Args:
        ax:    Matplotlib Axes to draw on.
        mu:    2-D mean vector of shape (2,).
        Sigma: 2x2 covariance matrix.
        color: Ellipse fill and edge colour.
        n_std: Number of standard deviations for the ellipse radius.
        alpha: Ellipse fill transparency.
        lw:    Edge line width.
    '''
    evals, evecs = np.linalg.eigh(Sigma)
    evals        = np.maximum(evals, 0.0)   # guard against tiny negatives
    angle        = np.degrees(np.arctan2(evecs[1, -1], evecs[0, -1]))
    width        = 2.0 * n_std * np.sqrt(evals[-1])
    height       = 2.0 * n_std * np.sqrt(evals[0])
    ellipse = Ellipse(
        xy=mu, width=width, height=height, angle=angle,
        facecolor=color, alpha=alpha, edgecolor=color,
        linewidth=lw, zorder=3,
    )
    ax.add_patch(ellipse)
    ax.plot(*mu, marker='x', color=color, markersize=9, markeredgewidth=2, zorder=4)


def plot_gmm_clusters(
    ax: plt.Axes,
    X: np.ndarray,
    gmm: GaussianMixtureModel,
    y_true: np.ndarray | None = None,
    title: str = 'GMM Clusters',
    show_ellipses: bool = True,
) -> None:
    '''Scatter-plot data coloured by GMM predicted component with ellipses.

    Args:
        ax:             Matplotlib Axes.
        X:              Data of shape (n, 2).
        gmm:            Fitted GaussianMixtureModel instance.
        y_true:         Optional ground-truth labels for ARI annotation.
        title:          Plot title.
        show_ellipses:  Whether to overlay Gaussian ellipses.
    '''
    labels = gmm.predict(X)
    for k in range(gmm.n_components):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=[COLORS[k % len(COLORS)]],
                   alpha=0.5, s=20, label=f'k={k}')
        if show_ellipses and gmm.mus_ is not None:
            plot_gmm_ellipse(ax, gmm.mus_[k], gmm.Sigmas_[k],
                             color=COLORS[k % len(COLORS)])
    if y_true is not None:
        ari = adjusted_rand_score(y_true, labels)
        ax.set_title(f'{title}\nARI={ari:.3f}', fontsize=11)
    else:
        ax.set_title(title, fontsize=11)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

In [ ]:
# ── EM convergence snapshots ──────────────────────────────────────────────────
result_hist = run_em_with_history(
    X, K=K_TRUE, init_method='random', random_state=SEED + 7,
    snapshot_iters=(0, 1, 3, 10, 50)
)

snap_iters = sorted(result_hist['snapshots'].keys())
print(f'Snapshots saved at iterations: {snap_iters}')
print(f'Total EM iterations           : {result_hist["n_iter"]}')

# ── Plot snapshots ────────────────────────────────────────────────────────────
n_snaps = len(snap_iters)
fig, axes = plt.subplots(1, n_snaps, figsize=(4 * n_snaps, 4))

for idx, it in enumerate(snap_iters):
    snap = result_hist['snapshots'][it]
    ax   = axes[idx]

    # Scatter coloured by soft responsibility (argmax)
    hard_labels = np.argmax(snap['log_R'], axis=1)
    for k in range(K_TRUE):
        mask = hard_labels == k
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=[COLORS[k]], alpha=0.4, s=12)

    # Draw ellipses at this iteration
    for k in range(K_TRUE):
        try:
            plot_gmm_ellipse(
                ax, snap['mus'][k], snap['Sigmas'][k],
                color=COLORS[k], n_std=1.5, alpha=0.2
            )
        except Exception:
            pass   # skip degenerate ellipses early in training

    ll = snap['ll']
    ax.set_title(f'Iter {it}\nlog-lik={ll:.0f}', fontsize=10)
    ax.set_xlabel('F1')
    if idx == 0:
        ax.set_ylabel('F2')
    ax.set_xlim(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3)
    ax.set_ylim(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3)

fig.suptitle('EM Convergence: Parameter Evolution Across Iterations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Log-likelihood curve with annotated snapshots ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(result_hist['log_likelihoods'], '-', color='steelblue', lw=2)
for it in snap_iters:
    ll = result_hist['snapshots'][it]['ll']
    ax.axvline(it, color='tomato', linestyle=':', alpha=0.7)
    ax.plot(it, ll, 'o', color='tomato', ms=8)
ax.set_xlabel('EM Iteration')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Log-Likelihood Trajectory (monotonically non-decreasing)', fontsize=12)
plt.tight_layout()
plt.show()

### Covariance Type Ablation

Full GMMs fit a complete $d \times d$ covariance matrix per component. Restricting the covariance reduces model capacity but improves robustness when $n$ is small:
- **Full:** $K \cdot d(d+1)/2$ covariance parameters — most expressive
- **Diagonal:** $K \cdot d$ parameters — assumes independent features
- **Spherical:** $K$ parameters — isotropic covariance (like K-Means)

In [ ]:
def m_step_diagonal(
    X: np.ndarray,
    log_R: np.ndarray,
    reg: float = REG_COV,
) -> tuple:
    '''M-step assuming diagonal (independent features) covariance.

    Args:
        X:     Data matrix of shape (n, d).
        log_R: Log-responsibility matrix of shape (n, K).
        reg:   Regularisation for variance values.

    Returns:
        log_pi: Updated log mixing weights of shape (K,).
        mus:    Updated list of K means, each of shape (d,).
        Sigmas: Updated list of diagonal covariance matrices, each (d, d).
    '''
    n, d = X.shape
    K    = log_R.shape[1]
    R    = np.exp(log_R)
    N_k  = np.maximum(R.sum(axis=0), 1e-10)
    log_pi = np.log(N_k) - np.log(n)
    mus = [(R[:, k] @ X) / N_k[k] for k in range(K)]
    Sigmas = []
    for k in range(K):
        diff   = X - mus[k]               # (n, d)
        w      = R[:, k] / N_k[k]         # (n,)
        var_k  = np.sum(w[:, None] * diff ** 2, axis=0) + reg   # (d,)
        Sigmas.append(np.diag(var_k))
    return log_pi, mus, Sigmas


def m_step_spherical(
    X: np.ndarray,
    log_R: np.ndarray,
    reg: float = REG_COV,
) -> tuple:
    '''M-step assuming spherical (isotropic) covariance.

    Args:
        X:     Data matrix of shape (n, d).
        log_R: Log-responsibility matrix of shape (n, K).
        reg:   Regularisation for variance value.

    Returns:
        log_pi: Updated log mixing weights of shape (K,).
        mus:    Updated list of K means, each of shape (d,).
        Sigmas: Updated list of isotropic covariance matrices, each (d, d).
    '''
    n, d = X.shape
    K    = log_R.shape[1]
    R    = np.exp(log_R)
    N_k  = np.maximum(R.sum(axis=0), 1e-10)
    log_pi = np.log(N_k) - np.log(n)
    mus = [(R[:, k] @ X) / N_k[k] for k in range(K)]
    Sigmas = []
    for k in range(K):
        diff    = X - mus[k]                         # (n, d)
        w       = R[:, k] / N_k[k]                  # (n,)
        sigma_sq = np.sum(w[:, None] * diff**2) / d + reg   # scalar
        Sigmas.append(sigma_sq * np.eye(d))
    return log_pi, mus, Sigmas


def run_em_cov_type(
    X: np.ndarray,
    K: int,
    cov_type: str = 'full',
    max_iter: int = MAX_ITER_EM,
    tol: float = TOL_EM,
    random_state: int = SEED,
    reg: float = REG_COV,
) -> dict:
    '''Run EM with a specified covariance type.

    Args:
        X:            Data matrix of shape (n, d).
        K:            Number of components.
        cov_type:     One of 'full', 'diagonal', 'spherical'.
        max_iter:     Maximum EM iterations.
        tol:          Log-likelihood convergence tolerance.
        random_state: Random seed.
        reg:          Covariance regularisation.

    Returns:
        Dict with keys log_pi, mus, Sigmas, log_likelihoods, final_ll.
    '''
    m_step_fn = {'full': m_step, 'diagonal': m_step_diagonal,
                 'spherical': m_step_spherical}[cov_type]
    log_pi, mus, Sigmas = initialize_gmm_params(
        X, K, method='kmeans', random_state=random_state, reg=reg
    )
    lls     = []
    prev_ll = -np.inf
    for it in range(max_iter):
        log_R, ll = e_step(X, log_pi, mus, Sigmas, reg=reg)
        lls.append(ll)
        if it > 0 and abs(ll - prev_ll) < tol:
            break
        prev_ll = ll
        log_pi, mus, Sigmas = m_step_fn(X, log_R, reg=reg)
    _, final_ll = e_step(X, log_pi, mus, Sigmas, reg=reg)
    return {'log_pi': log_pi, 'mus': mus, 'Sigmas': Sigmas,
            'log_likelihoods': lls, 'final_ll': final_ll,
            'n_iter': len(lls)}


cov_types   = ['full', 'diagonal', 'spherical']
cov_results = {}
print('Covariance type comparison (K=4):')
for ct in cov_types:
    res = run_em_cov_type(X, K=K_TRUE, cov_type=ct, random_state=SEED)
    cov_results[ct] = res
    labels_ct = np.argmax(np.exp(
        np.array([e_step(X, res['log_pi'], res['mus'], res['Sigmas'])[0]])[0]
    ), axis=1)
    # recompute labels cleanly
    log_R_ct, _ = e_step(X, res['log_pi'], res['mus'], res['Sigmas'])
    labels_ct   = np.argmax(log_R_ct, axis=1)
    ari_ct      = adjusted_rand_score(y_true, labels_ct)
    nmi_ct      = normalized_mutual_info_score(y_true, labels_ct)
    print(f'  {ct:10s}  final_ll={res["final_ll"]:.1f}  '
          f'n_iter={res["n_iter"]}  ARI={ari_ct:.3f}  NMI={nmi_ct:.3f}')

# ── Side-by-side scatter ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ct in zip(axes, cov_types):
    res      = cov_results[ct]
    log_R_ct, _ = e_step(X, res['log_pi'], res['mus'], res['Sigmas'])
    labels_ct   = np.argmax(log_R_ct, axis=1)
    ari_ct      = adjusted_rand_score(y_true, labels_ct)
    for k in range(K_TRUE):
        mask = labels_ct == k
        ax.scatter(X[mask, 0], X[mask, 1], c=[COLORS[k]], alpha=0.5, s=15)
    for k in range(K_TRUE):
        plot_gmm_ellipse(ax, res['mus'][k], res['Sigmas'][k],
                         color=COLORS[k], n_std=1.5, alpha=0.15)
    ax.set_title(f'{ct.capitalize()} Cov\nARI={ari_ct:.3f}', fontsize=11)
    ax.set_xlabel('Feature 1')
    if ct == 'full':
        ax.set_ylabel('Feature 2')
fig.suptitle('Covariance Type Ablation (K=4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── GMM density heatmap & sampling ────────────────────────────────────────────
gmm_full = GaussianMixtureModel(n_components=K_TRUE, n_init=N_INIT, random_state=SEED)
gmm_full.fit(X)

# Build a grid for density evaluation
x1_lim = (X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
x2_lim = (X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
xx1, xx2 = np.meshgrid(
    np.linspace(*x1_lim, 200),
    np.linspace(*x2_lim, 200),
)
X_grid   = np.column_stack([xx1.ravel(), xx2.ravel()])

# Evaluate log-density on the grid
log_w_grid = np.zeros((len(X_grid), K_TRUE))
for k in range(K_TRUE):
    log_w_grid[:, k] = (
        gmm_full.log_pi_[k]
        + log_multivariate_gaussian_pdf(X_grid, gmm_full.mus_[k],
                                        gmm_full.Sigmas_[k])
    )
log_density = logsumexp(log_w_grid, axis=1).reshape(xx1.shape)

# Draw samples from the fitted GMM
X_samp, comp_samp = gmm_full.sample(n_samples=300, random_state=SEED + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
cf = ax.contourf(xx1, xx2, log_density, levels=30, cmap='viridis', alpha=0.85)
fig.colorbar(cf, ax=ax, label='Log-Density')
ax.scatter(X[:, 0], X[:, 1], c='white', alpha=0.3, s=10, label='Training data')
for k in range(K_TRUE):
    plot_gmm_ellipse(ax, gmm_full.mus_[k], gmm_full.Sigmas_[k],
                     color='white', n_std=2.0, alpha=0.0, lw=2)
ax.set_title('Fitted GMM Log-Density', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)

ax = axes[1]
for k in range(K_TRUE):
    mask = comp_samp == k
    ax.scatter(X_samp[mask, 0], X_samp[mask, 1],
               c=[COLORS[k]], alpha=0.6, s=25, label=f'k={k}')
ax.scatter(X[:, 0], X[:, 1], c='grey', alpha=0.15, s=10, label='Training')
ax.set_title('Samples Drawn from Fitted GMM', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=8)

fig.suptitle('GMM as a Generative Density Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Clustering metrics summary ────────────────────────────────────────────────
gmm_pred  = gmm_full.predict(X)
ari_final = adjusted_rand_score(y_true, gmm_pred)
nmi_final = normalized_mutual_info_score(y_true, gmm_pred)
sil_final = silhouette_score(X, gmm_pred)

print('Final clustering metrics (Our GMM, K=4):')
print(f'  ARI (Adjusted Rand Index)     : {ari_final:.4f}  (1.0 = perfect)')
print(f'  NMI (Normalised Mutual Info)  : {nmi_final:.4f}  (1.0 = perfect)')
print(f'  Silhouette score              : {sil_final:.4f}  (-1..1, higher=better)')
print(f'  Log-likelihood                : {gmm_full.log_likelihood_:.2f}')
print(f'  BIC                           : {gmm_full.bic(X):.2f}')
print(f'  AIC                           : {gmm_full.aic(X):.2f}')

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **GMMs generalise K-Means** by using soft (probabilistic) cluster assignments and fitting per-component covariances. K-Means is the hard-assignment, spherical-covariance limit of EM.

2. **EM monotonically increases the log-likelihood** but converges to a local maximum. Multiple restarts (`n_init`) and K-Means initialisation reduce the risk of poor solutions.

3. **BIC and AIC penalise model complexity** differently: BIC ($p \log n$) is more conservative and tends to select fewer components. Both correctly identify the true K on well-separated data.

4. **Full covariance captures more structure** (oriented clusters, different scales) at the cost of more parameters. Diagonal and spherical covariances are regularised alternatives when $n \ll d$.

5. **Numerical stability requires care**: log-sum-exp prevents underflow in the E-step; Cholesky decomposition provides stable log-determinants; diagonal regularisation keeps covariances positive definite.

### What's Next

→ **03-07 (Anomaly Detection)** uses the Gaussian density estimation we built here as the foundation for statistical anomaly scoring.

→ **03-10 (Bayesian Inference)** extends the probabilistic perspective: the variational EM algorithm (used in VAEs, Module 11) generalises the E-step from exact posteriors to approximate distributions.

### Going Further

- Bishop (2006) *Pattern Recognition and Machine Learning* — Chapter 9: thorough EM derivation and variational EM.
- Reynolds (2009) *Gaussian Mixture Models* — tutorial with applications in speaker identification.
- Kingma & Welling (2014) *Auto-Encoding Variational Bayes* — extends EM to neural network decoders.